# Milestone 3

In [1]:
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "18g") \
    .config("spark.executor.instances", "7") \
    .getOrCreate()

In [2]:
PARQUET_PATH = "data/final_preprocessed"

df = spark.read.parquet(PARQUET_PATH)

In [3]:
from pyspark.ml.feature import VectorAssembler

SEED = 42
sp = max(200, spark.sparkContext.defaultParallelism * 4)
spark.conf.set("spark.sql.shuffle.partitions", str(sp))

ml_df = df.select(F.col("REALINCTOT").alias("label"), F.col("EDUC").cast("double").alias("EDUC"), "AGE_Z", "STATE_OH", "SEX_OH", "RACE_OH")
assembler = VectorAssembler(inputCols=["EDUC", "AGE_Z", "STATE_OH", "SEX_OH", "RACE_OH"], outputCol="features")
ml_df = assembler.transform(ml_df)

train_df, val_df, test_df = ml_df.randomSplit([0.70, 0.15, 0.15], seed=SEED)

In [4]:
import requests
import pandas as pd

# Get the active Spark Context and URL
sc = spark.sparkContext
url = f"{sc.uiWebUrl}/api/v1/applications/{sc.applicationId}/executors"

# Fetch the executor data from the API
response = requests.get(url)
executors = response.json()

# Format into a readable DataFrame
df = pd.DataFrame(executors)[['id', 'totalCores', 'maxMemory', 'activeTasks', 'isActive']]
df['maxMemory_GB'] = (df['maxMemory'] / (1024**3)).round(2)
df

,id,totalCores,maxMemory,activeTasks,isActive,maxMemory_GB
0,driver,8,1099746508,0,True,1.02


In [5]:
print("master:", spark.sparkContext.master)
print("appId:", spark.sparkContext.applicationId)
print("executor.instances:", spark.conf.get("spark.executor.instances", "NA"))
print("executor.memory:", spark.conf.get("spark.executor.memory", "NA"))

master: local[*]
appId: local-1778603026812
executor.instances: 7
executor.memory: 18g


In [6]:
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator

rf = RandomForestRegressor(labelCol="label", featuresCol="features", predictionCol="prediction", numTrees=20, maxDepth=10, seed=SEED)
model_baseline = rf.fit(train_df)

ev_rmse = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="rmse")
ev_mae = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="mae")
ev_r2 = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="r2")

pred = model_baseline.transform(train_df)
print("train_baseline rmse mae r2", ev_rmse.evaluate(pred), ev_mae.evaluate(pred), ev_r2.evaluate(pred))
pred = model_baseline.transform(val_df)
print("val_baseline rmse mae r2", ev_rmse.evaluate(pred), ev_mae.evaluate(pred), ev_r2.evaluate(pred))
pred = model_baseline.transform(test_df)
print("test_baseline rmse mae r2", ev_rmse.evaluate(pred), ev_mae.evaluate(pred), ev_r2.evaluate(pred))

train_baseline rmse mae r2 31627.076569655525 15340.427044772552 0.25120058259438227
val_baseline rmse mae r2 31647.33606421245 15340.852121658128 0.25134928387479294
test_baseline rmse mae r2 31620.073788916903 15330.536140079486 0.25124269126723286


In [ ]:
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator

rf2 = RandomForestRegressor(labelCol="label", featuresCol="features", predictionCol="prediction", numTrees=30, maxDepth=12, seed=SEED)
model_rf_hp = rf2.fit(train_df)

pred = model_rf_hp.transform(val_df)

In [5]:
ev_rmse = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="rmse")
ev_mae = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="mae")
ev_r2 = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="r2")

In [6]:
print("val_rf30_d12 rmse mae r2", ev_rmse.evaluate(pred), ev_mae.evaluate(pred), ev_r2.evaluate(pred))
pred = model_rf_hp.transform(test_df)
print("test_rf30_d12 rmse mae r2", ev_rmse.evaluate(pred), ev_mae.evaluate(pred), ev_r2.evaluate(pred))

val_rf30_d12 rmse mae r2 31587.003914396468 15260.154559595332 0.25420100279683766
test_rf30_d12 rmse mae r2 31559.9525723881 15249.530292920308 0.25408730222752307


In [9]:
pred = model_rf_hp.transform(train_df)
print("train_rf30_d12 rmse mae r2", ev_rmse.evaluate(pred), ev_mae.evaluate(pred), ev_r2.evaluate(pred))

train_rf30_d12 rmse mae r2 31565.508469408232 15258.63254374953 0.2541131049832106
